# library

In [1]:
import pandas as pd
import pandas as pd
from groq import Groq
from tqdm import tqdm  # progress bar
import re

# loading data

In [28]:
medquad_df = pd.read_csv('medquad.csv')

# checking df properties

In [29]:
medquad_df.describe()

,question,answer,source,focus_area
count,16412,16407,16412,16398
unique,14984,15817,9,5126
top,What causes Causes of Diabetes ?,This condition is inherited in an autosomal re...,GHR,Breast Cancer
freq,20,348,5430,53


In [30]:
medquad_df.isna().sum()

question       0
answer         5
source         0
focus_area    14
dtype: int64

# Dropping null answers rows.

In [31]:
drop_null_answers_df = medquad_df.dropna(subset=['answer'])

In [32]:
drop_null_answers_df.shape

(16407, 4)

# lowercase entire df

In [33]:
lowercase_df = drop_null_answers_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

/var/folders/bl/c0pvtpr55y3477p6dl_6s5kh0000gn/T/ipykernel_3471/3527642856.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  lowercase_df = drop_null_answers_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


# dropping dupliacted rows in entire df

In [34]:
drop_duplicated_df = lowercase_df.drop_duplicates(subset=['question', 'answer', 'source', 'focus_area'])

In [35]:
drop_duplicated_df.shape

(16359, 4)

# removing (are) in “What is (are) Glaucoma ?” type of statements

In [36]:
drop_duplicated_df["question"] = drop_duplicated_df["question"].str.replace(r"\(.*?\)", "", regex=True).str.strip()

/var/folders/bl/c0pvtpr55y3477p6dl_6s5kh0000gn/T/ipykernel_3471/325347466.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  drop_duplicated_df["question"] = drop_duplicated_df["question"].str.replace(r"\(.*?\)", "", regex=True).str.strip()


# stripping answer column

In [37]:
drop_duplicated_df["answer"] = drop_duplicated_df["answer"].str.strip()

/var/folders/bl/c0pvtpr55y3477p6dl_6s5kh0000gn/T/ipykernel_3471/3061686933.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  drop_duplicated_df["answer"] = drop_duplicated_df["answer"].str.strip()


# Replace multiple spaces/tabs/newlines inside text with a single space, Remove leading & trailing spaces (including tabs, line breaks)

In [38]:
drop_duplicated_df["question"] = drop_duplicated_df["question"].str.replace(r"\?\s*\?+", "", regex=True)
# Step 2: remove unwanted symbols but keep %, mg, °C, ?
drop_duplicated_df["question"] = drop_duplicated_df["question"].str.replace(r"[^a-zA-Z0-9\s%\°\?]", "", regex=True)
# Step 3: clean spaces
drop_duplicated_df["question"] = drop_duplicated_df["question"].str.strip().str.replace(r"\s+", " ", regex=True)

/var/folders/bl/c0pvtpr55y3477p6dl_6s5kh0000gn/T/ipykernel_3471/3375683636.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  drop_duplicated_df["question"] = drop_duplicated_df["question"].str.replace(r"\?\s*\?+", "", regex=True)
/var/folders/bl/c0pvtpr55y3477p6dl_6s5kh0000gn/T/ipykernel_3471/3375683636.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  drop_duplicated_df["question"] = drop_duplicated_df["question"].str.replace(r"[^a-zA-Z0-9\s%\°\?]", "", regex=True)
/var/folders/bl/c0pvtpr55y3477p6dl_6s

# Group by on unique questions and join all answers of same questions

In [14]:
df_grouped = drop_duplicated_df.groupby("question", as_index=False).agg({
    "answer": lambda x: " ".join(x.astype(str))
})

In [15]:
df_grouped.to_csv("unique_question_answers.csv", index=False)

print("Before merging:", len(drop_duplicated_df))
print("After merging:", len(df_grouped))

Before merging: 16359
After merging: 14322


# Used groq api script to shorten the answers and remove garbage like emails, URLs, and unrelated material. (only 1000 rows)

In [ ]:
# Initialize Groq client
client = Groq(api_key="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")

# Load dataset
df = pd.read_csv("unique_question_answers.csv")

# Only take first 1000 rows for now
df = df[2001:3000].head(1000)

# Function to clean answer
def clean_answer(question, answer):
    prompt = f"""
You are given a question and its long answer. 
Your task:
    - Shorten the answer but keep it meaningful according to question. 
    - Do not use any outside knowledge (only the provided text). 
    - Remove garbage like emails, URLs, and unrelated material. 
    - Make sure the answer is still relevant to the question. 
    - Do not add any extra phrases like "Here is the answer" or "Cleaned response".
    - Do not repeat the question in the answer
    - Return only the final cleaned answer text.

Question: {question}
Answer: {answer}
"""
    try:
        response = client.chat.completions.create(
            model="llama3-8b-8192",  # ⚡ FASTER model (smaller than 70b)
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
            max_tokens=512
        )
        cleaned = response.choices[0].message.content.strip()

        # Post-cleaning: remove unwanted prefixes
        cleaned = re.sub(r"(?i)^here is.*?:", "", cleaned).strip()
        cleaned = re.sub(rf"(?i){re.escape(question)}", "", cleaned).strip()

        return cleaned
    except Exception as e:
        print("Error:", e)
        return answer

# Apply with progress bar
df["cleaned_answer"] = [
    clean_answer(q, a) for q, a in tqdm(zip(df["question"], df["answer"]), total=len(df))
]

# Save
df.to_csv("final_cleaned_answers_1000_rows.csv", index=False)
print("✅ Cleaning completed for 1000 rows and saved to final_cleaned_answers_1000_rows.csv")

# copy 1k rows

In [17]:
merged_answers_df = df_grouped.head(1000).copy()

In [2]:
final_cleaned_answers_df = pd.read_csv("final_cleaned_answers.csv")

In [3]:
final_cleaned_answers_df

,question,answer,cleaned_answer
0,are certain people at risk of getting vancomyc...,on this page general information what is vanco...,Certain people are at increased risk of gettin...
1,are there complications from botulism?,botulism can result in death due to respirator...,Botulism can result in death from respiratory ...
2,do you have information about a1c,summary : a1c is a blood test for type 2 diabe...,A1c is a blood test that measures your average...
3,do you have information about acupuncture,summary : acupuncture has been practiced in ch...,Acupuncture has been practiced in China and ot...
4,do you have information about adoption,summary : adoption brings a child born to othe...,Adoption brings a child born to other parents ...
...,...,...,...
995,how many people are affected by methylmalonic ...,"the most common form of the condition, called ...","1 in 200,000 newborns worldwide are affected b..."
996,how many people are affected by mevalonate kin...,more than 200 people with mevalonate kinase de...,More than 200 people have been reported with m...
997,how many people are affected by microcephalic ...,"mopdii appears to be a rare condition, althoug...","appears to be a rare condition, although its p..."
998,how many people are affected by microcephalyca...,microcephaly-capillary malformation syndrome i...,about a dozen people have been diagnosed with ...


# replace the short answer column with long answer column 

In [ ]:
merged_answers_df['answer'] = final_cleaned_answers_df['cleaned_answer']

# lowercase entire df

In [20]:
merged_answers_lowercase_df = merged_answers_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

/var/folders/bl/c0pvtpr55y3477p6dl_6s5kh0000gn/T/ipykernel_3471/2106081058.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  merged_answers_lowercase_df = merged_answers_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


# cleaning answer column

In [21]:
# Function to clean text
def clean_text(text):
    if pd.isnull(text):  # handle NaN values
        return text
    # Keep only alphabets, numbers, and medical terms-related chars
    cleaned = re.sub(r"[^a-zA-Z0-9\s\-/%]", "", str(text))  
    # Remove extra spaces
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned

In [ ]:
merged_answers_lowercase_df["answer"] = merged_answers_lowercase_df["answer"].apply(clean_text)

In [24]:
merged_answers_lowercase_df.to_csv("final_dataset_cleaned.csv", index=False)

# converting to pickle 

In [42]:
merged_answers_lowercase_df.to_pickle("final_dataset_cleaned.pkl")